In [4]:
import psycopg
import os
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import json
from pathlib import Path
from pgvector.psycopg import register_vector

In [5]:
load_dotenv() #library used to read the env file
DDL = """
CREATE EXTENSION IF NOT EXISTS vector;

DROP TABLE IF EXISTS chunks CASCADE;

CREATE TABLE chunks (
    chunk_id        TEXT PRIMARY KEY,
    text            TEXT NOT NULL,
    embedding       vector(384) NOT NULL,
    jurisdiction    TEXT NOT NULL,
    act             TEXT,
    section_number  TEXT,
    section_title   TEXT,
    part            TEXT,
    division        TEXT,
    schedule        TEXT,
    source_url      TEXT,
    version         TEXT,
    downloaded_date TEXT
);

CREATE INDEX chunks_embedding_idx ON chunks
        USING hnsw (embedding vector_cosine_ops);

CREATE INDEX chunks_jurisdiction_idx ON chunks (jurisdiction);
"""

#This is the code snippet to run a DDL file
with psycopg.connect(os.getenv("DATABASE_URL")) as conn:
    with conn.cursor() as cur:
        cur.execute(DDL)
    conn.commit()

print("Table Created")

Table Created


In [6]:
#Checking if the file exists or not
with psycopg.connect(os.getenv("DATABASE_URL")) as conn:
    with conn.cursor() as cur:
        cur.execute("""
                    SELECT column_name,data_type
                    FROM information_schema.columns
                    WHERE table_name = 'chunks'
                    ORDER BY ordinal_position;
                    """)
        for col, dtype in cur.fetchall():
            print(f" {col:20s} {dtype}")

 chunk_id             text
 text                 text
 embedding            USER-DEFINED
 jurisdiction         text
 act                  text
 section_number       text
 section_title        text
 part                 text
 division             text
 schedule             text
 source_url           text
 version              text
 downloaded_date      text


In [7]:
#Embedding models
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print(f"Output dim: {model.get_sentence_embedding_dimension()}")

sample = model.encode("How much notice for rent increase?")
print(f"Sample shape: {sample.shape}")

Output dim: 384
Sample shape: (384,)


In [8]:
CHUNKS_JSONL = Path("/Users/varunchandrashekar/Tenantmate/code/data/processed/nsw_chunks.jsonl")

with open(CHUNKS_JSONL) as f:
    chunks = [json.loads(line) for line in f]

print(f"Loaded {len(chunks)} chunks")

texts = [c["text"] for c in chunks]
print("Embedding... (~30 seconds on CPU)")
embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)

print(f"Embeddings shape: {embeddings.shape}")    # f-string now, runs AFTER encode
print(f"Total: {len(chunks)}")
print(f"With schedule: {sum(1 for c in chunks if c.get('schedule'))}")
print(f"First and last chunk_id: {chunks[0]['chunk_id']}, {chunks[-1]['chunk_id']}")

Loaded 328 chunks
Embedding... (~30 seconds on CPU)


Batches: 100%|██████████| 11/11 [00:03<00:00,  3.02it/s]

Embeddings shape: (328, 384)
Total: 328
With schedule: 52
First and last chunk_id: NSW-RTA2010-s1, NSW-RTA2010-Schedule2-s36


In [9]:
INSERT_SQL = """
INSERT INTO chunks (
    chunk_id, text, embedding,
    jurisdiction, act, section_number, section_title,
    part, division, schedule,
    source_url, version, downloaded_date
) VALUES (
    %s, %s, %s,
    %s, %s, %s, %s,
    %s, %s, %s,
    %s, %s, %s
)
ON CONFLICT (chunk_id) DO UPDATE SET
    text = EXCLUDED.text,
    embedding = EXCLUDED.embedding;
"""

with psycopg.connect(os.getenv("DATABASE_URL")) as conn:
    register_vector(conn)
    with conn.cursor() as cur:
        for chunk, vector in zip(chunks, embeddings):
            cur.execute(INSERT_SQL, (
                chunk["chunk_id"],
                chunk["text"],
                vector,
                chunk["jurisdiction"],
                chunk.get("act"),
                chunk.get("section_number"),
                chunk.get("section_title"),
                chunk.get("part"),
                chunk.get("division"),
                chunk.get("schedule"),
                chunk.get("source_url"),
                chunk.get("version"),
                chunk.get("downloaded_date"),
            ))
    conn.commit()

print(f"Inserted {len(chunks)} chunks")

Inserted 328 chunks


In [10]:
def search(query:str, k: int = 5):
    """Embed query, return top-k nearest chunks by cosine similarity"""
    q_vec = model.encode(query)

    SQL = """
    SELECT chunk_id, section_title, part,
        1 - (embedding <=> %s) AS similarity,
        text
    FROM chunks
    ORDER BY embedding <=> %s
    LIMIT %s;
"""
    with psycopg.connect(os.getenv("DATABASE_URL")) as conn:
        register_vector(conn)
        with conn.cursor() as cur:
            cur.execute(SQL, (q_vec, q_vec, k))
            return cur.fetchall()

#Testing with some questions
questions = [
    "How much notice is required for a rent increase?",
    "Can my landlord enter the property without notice?",
    "What are the rules around getting my rental bond back?",
    "Can I have a pet in my rental?",
]

for q in questions:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    print('='*70)
    results = search(q, k=3)
    for chunk_id, title, part, sim, text in results:
        print(f"\n  [{sim:.3f}] {chunk_id} — {title}")
        print(f"          {part}")
        print(f"          {text[:150].strip()}...")



Q: How much notice is required for a rent increase?

  [0.803] NSW-RTA2010-s41 — Rent increases
          Part 3 Rights and obligations of landlords and tenants
          (1) The rent payable under a residential tenancy agreement may be increased only if—
(a) the tenant is given a written notice by the landlord or the l...

  [0.783] NSW-RTA2010-s99 — Rent increases during long-term fixed term leases—termination notice by tenant
          Part 5 Termination of residential tenancy agreements
          (1) This section applies to a fixed term agreement for a fixed term of 2 years or more.
(2) A tenant may give a termination notice on the ground that...

  [0.743] NSW-RTA2010-s44 — Tenant’s remedies for excessive rent
          Part 3 Rights and obligations of landlords and tenants
          (1) Excessive rent orders The Tribunal may, on the application of a tenant, make any of the
following orders—
(a) an order that a rent increase under...

Q: Can my landlord enter the property without

For each question, the top results should be topically relevant sections. Not perfect, but recognisably about the right thing:

"rent increase" → sections about rent / rent increase rules
"landlord enter" → sections about access / entry by landlord
"rental bond" → sections about bonds / Schedule 1 (Rental Bond Board)
"pet" → sections about animals / pets (this is one of the new 2024 reform areas)

Two notes on the SQL:

embedding <=> %s — pgvector's cosine distance operator
1 - distance = similarity — higher = more relevant (between 0 and 1)

We are alternating the vector database with tsvector 

In [14]:
load_dotenv()

DDL = """
ALTER TABLE chunks
    ADD COLUMN IF NOT EXISTS text_tsv tsvector
    GENERATED ALWAYS AS (to_tsvector('english', text)) STORED;

CREATE INDEX IF NOT EXISTS chunks_text_tsv_idx
    ON chunks USING GIN (text_tsv);
"""

with psycopg.connect(os.getenv("DATABASE_URL")) as conn:
    with conn.cursor() as cur:
        cur.execute(DDL)
    conn.commit()

print("tsvector column added and indexed")

tsvector column added and indexed


In [12]:
with psycopg.connect(os.getenv("DATABASE_URL")) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM chunks;")
        print(f"Rows: {cur.fetchone()[0]}")

Rows: 328
